# T4D Response V2 - Phase 2 Screening And Cohorts

Goal: remove weak stations and weak station-years, describe the cuts clearly, and create one simple train/holdout split table.

Outputs:
- `t4d.v2.phase2.station_summary.csv`
- `t4d.v2.phase2.station_year_summary.csv`
- `t4d.v2.phase2.daily_screened.csv`
- `t4d.v2.phase2.cohort_keys.csv`

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme( style = 'whitegrid' )
pd.set_option( 'display.max_columns', 120 )
pd.set_option( 'display.max_rows', 80 )

phase_dir = Path( '../data/reference/response_v2' )
daily_path = phase_dir / 't4d.v2.phase1.daily_foundation.csv'
station_meta_path = phase_dir / 't4d.v2.phase1.station_metadata.csv'

station_summary_out_path = phase_dir / 't4d.v2.phase2.station_summary.csv'
station_year_out_path = phase_dir / 't4d.v2.phase2.station_year_summary.csv'
daily_screened_out_path = phase_dir / 't4d.v2.phase2.daily_screened.csv'
cohort_out_path = phase_dir / 't4d.v2.phase2.cohort_keys.csv'

daily = pd.read_csv( daily_path, parse_dates = [ 'date' ] )
station_meta = pd.read_csv( station_meta_path )

In [ ]:
# Summarize stations first.
# Then summarize station-years.
# Keep the rules plain enough that you can read them like a checklist.
station_summary = ( 
    daily
    .groupby( [ 'station_key', 'station_code', 'region', 'station', 'region_name', 'station_name' ], as_index = False )
    .agg( 
        first_date = ( 'date', 'min' ),
        last_date = ( 'date', 'max' ),
        n_daily_rows = ( 'date', 'size' ),
        n_years = ( 'year', 'nunique' ),
        n_temp_days = ( 'water_temp', lambda values: int( values.notna( ).sum( ) ) ),
        n_oxygen_days = ( 'oxygen', lambda values: int( values.notna( ).sum( ) ) ) if 'oxygen' in daily.columns else ( 'date', 'size' ),
    )
)

station_year_summary = ( 
    daily
    .groupby( [ 'station_key', 'station_code', 'region', 'station', 'region_name', 'station_name', 'year' ], as_index = False )
    .agg( 
        n_days = ( 'date', 'size' ),
        n_temp_days = ( 'water_temp', lambda values: int( values.notna( ).sum( ) ) ),
        n_oxygen_days = ( 'oxygen', lambda values: int( values.notna( ).sum( ) ) ) if 'oxygen' in daily.columns else ( 'date', 'size' ),
        n_any_months = ( 'month_num', 'nunique' ),
    )
)

warm_oxygen = ( 
    daily.loc[ daily[ 'is_warm_season' ] ]
    .groupby( [ 'station_key', 'year' ], as_index = False )
    .agg( 
        n_warm_oxygen_days = ( 'oxygen', lambda values: int( values.notna( ).sum( ) ) ) if 'oxygen' in daily.columns else ( 'date', 'size' ),
        n_warm_months = ( 'month_num', 'nunique' ),
    )
)
station_year_summary = station_year_summary.merge( warm_oxygen, on = [ 'station_key', 'year' ], how = 'left' ).fillna( 0 )

# This is the basic screen.
# First remove tiny stations.
# Then remove weak years inside otherwise decent stations.
min_station_years = 5
min_station_temp_days = 365 * 3
min_year_temp_days = 120
min_year_months = 6
min_warm_oxygen_days = 45
min_warm_oxygen_months = 3

station_summary[ 'keep_station' ] = ( 
    ( station_summary[ 'n_years' ] >= min_station_years )
    & ( station_summary[ 'n_temp_days' ] >= min_station_temp_days )
)
station_year_summary[ 'keep_year' ] = ( 
    ( station_year_summary[ 'n_temp_days' ] >= min_year_temp_days )
    & ( station_year_summary[ 'n_any_months' ] >= min_year_months )
)
station_year_summary[ 'keep_warm_oxygen_year' ] = ( 
    station_year_summary[ 'keep_year' ]
    & ( station_year_summary[ 'n_warm_oxygen_days' ] >= min_warm_oxygen_days )
    & ( station_year_summary[ 'n_warm_months' ] >= min_warm_oxygen_months )
)

station_summary[ 'screen_note' ] = np.where( station_summary[ 'keep_station' ], 'keep', 'drop_small_station' )
station_year_summary[ 'screen_note' ] = np.where( station_year_summary[ 'keep_year' ], 'keep', 'drop_sparse_year' )

print( 'station thresholds:' )
print( f'  min station years = {min_station_years}' )
print( f'  min station water-temp days = {min_station_temp_days}' )
print( 'year thresholds:' )
print( f'  min year water-temp days = {min_year_temp_days}' )
print( f'  min year active months = {min_year_months}' )
print( f'  min warm oxygen days = {min_warm_oxygen_days}' )
print( f'  min warm oxygen months = {min_warm_oxygen_months}' )
print( f'stations kept = {int( station_summary[ "keep_station" ].sum( ) )} / {len( station_summary )}' )
print( f'station-years kept = {int( station_year_summary[ "keep_year" ].sum( ) )} / {len( station_year_summary )}' )
print( f'warm oxygen years kept = {int( station_year_summary[ "keep_warm_oxygen_year" ].sum( ) )} / {len( station_year_summary )}' )

display( station_summary.head( 10 ) )
display( station_year_summary.head( 10 ) )

In [ ]:
# Build the screened daily table.
# Keep the split simple and deterministic.
# Sort stations inside each region and hold out the last 20 percent.
keep_station_keys = station_summary.loc[ station_summary[ 'keep_station' ], [ 'station_key' ] ].drop_duplicates( )
keep_station_year_keys = station_year_summary.loc[ station_year_summary[ 'keep_year' ], [ 'station_key', 'year', 'keep_warm_oxygen_year' ] ].drop_duplicates( )

daily_screened = daily.merge( keep_station_keys, on = 'station_key', how = 'inner' )
daily_screened = daily_screened.merge( keep_station_year_keys, on = [ 'station_key', 'year' ], how = 'inner' )

cohort_keys = ( 
    station_summary.loc[ station_summary[ 'keep_station' ], [ 'station_key', 'station_code', 'region', 'region_name', 'station_name' ] ]
    .sort_values( [ 'region', 'station_key' ] )
    .reset_index( drop = True )
)
cohort_keys[ 'region_rank' ] = cohort_keys.groupby( 'region' ).cumcount( )
cohort_keys[ 'region_n' ] = cohort_keys.groupby( 'region' )[ 'station_key' ].transform( 'size' )
cohort_keys[ 'region_frac' ] = ( cohort_keys[ 'region_rank' ] + 1 ) / cohort_keys[ 'region_n' ]
cohort_keys[ 'cohort' ] = np.where( cohort_keys[ 'region_frac' ] <= 0.80, 'train', 'holdout' )

daily_screened = daily_screened.merge( cohort_keys[ [ 'station_key', 'cohort' ] ], on = 'station_key', how = 'left' )

print( 'screened daily rows:', len( daily_screened ) )
print( 'train stations:', int( ( cohort_keys[ 'cohort' ] == 'train' ).sum( ) ) )
print( 'holdout stations:', int( ( cohort_keys[ 'cohort' ] == 'holdout' ).sum( ) ) )

display( cohort_keys.head( 12 ) )

In [ ]:
# Keep the charts direct.
# One shows station coverage.
# One shows how many years each cohort keeps.
fig, axes = plt.subplots( 1, 2, figsize = ( 16, 5 ) )

sns.histplot( data = station_summary, x = 'n_temp_days', hue = 'keep_station', bins = 30, ax = axes[ 0 ] )
axes[ 0 ].set_title( 'Phase 2 Station Water Temp Coverage' )
axes[ 0 ].set_xlabel( 'Observed water-temp days across full record' )
axes[ 0 ].set_ylabel( 'Count of stations' )

cohort_years = ( 
    daily_screened[ [ 'station_key', 'year', 'cohort' ] ]
    .drop_duplicates( )
    .groupby( [ 'cohort', 'year' ], as_index = False )
    .size( )
    .rename( columns = { 'size': 'n_station_years' } )
)
sns.lineplot( data = cohort_years, x = 'year', y = 'n_station_years', hue = 'cohort', marker = 'o', ax = axes[ 1 ] )
axes[ 1 ].set_title( 'Phase 2 Station-Years By Cohort' )
axes[ 1 ].set_xlabel( 'Year' )
axes[ 1 ].set_ylabel( 'Station-years kept' )

plt.tight_layout( )
plt.show( )

In [ ]:
station_summary.to_csv( station_summary_out_path, index = False )
station_year_summary.to_csv( station_year_out_path, index = False )
daily_screened.to_csv( daily_screened_out_path, index = False )
cohort_keys.to_csv( cohort_out_path, index = False )

print( f'saved: {station_summary_out_path}' )
print( f'saved: {station_year_out_path}' )
print( f'saved: {daily_screened_out_path}' )
print( f'saved: {cohort_out_path}' )